<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/02_fastq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FASTQ dosyasının içinde ne var?

Bu defterde ham veriyi ilk kez kendi gözünüzle göreceksiniz. Veri, Zenodo'daki paketimden iniyor; iki dosya indiriyoruz, biri yabani tip, biri knockout. Beş dakika sürüyor.

In [1]:
import gzip, urllib.request

for d in ['SRR384977_1M.fastq.gz', 'SRR384980_1M.fastq.gz']:
    urllib.request.urlretrieve('https://zenodo.org/records/22206786/files/' + d + '?download=1', d)
    print(d, 'indi')

SRR384977_1M.fastq.gz indi
SRR384980_1M.fastq.gz indi


## 1. İlk kayda bakıyorum

FASTQ'da her okuma dört satırdır: kimlik, dizi, artı işareti, kalite. Kalite satırındaki her karakter, dizideki aynı konumun güvenilirlik notudur.

In [2]:
with gzip.open('SRR384977_1M.fastq.gz', 'rt') as f:
    for i in range(8):
        print(f.readline().rstrip())

@SRR384977.1 ILLUMINA-A52086_0016:6:8:1061:16491/1
TGGACATGGAAGGCAACGAACANGATCNGGACCAGTGGATGATCNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
+
????????????????????????????????????????????????????????????????????????????
@SRR384977.2 ILLUMINA-A52086_0016:6:8:1061:13444/1
AAGTTGATGTGTTTAATTAGGTNTGATNTTGAAATTAGGATTGGNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
+
????????????????????????????????????????????????????????????????????????????


## 2. Kalite harflerini sayıya çeviriyorum

Bu, Phred ölçeği: karakterin ASCII kodundan 33 çıkarınca kalite skoru çıkar. Skor 30, "binde bir hata ihtimali" demek; 20, yüzde bir. Formül: hata olasılığı = 10^(-Q/10).

In [3]:
with gzip.open('SRR384977_1M.fastq.gz', 'rt') as f:
    f.readline(); f.readline(); f.readline()
    kalite = f.readline().rstrip()

for k in kalite[:10]:
    print(k, '->', ord(k) - 33)

? -> 30
? -> 30
? -> 30
? -> 30
? -> 30
? -> 30
? -> 30
? -> 30
? -> 30
? -> 30


## 3. İki dosyayı karşılaştırıyorum

Veri sayfasında bir gizem bırakmıştım: KO dosyaları aynı okuma sayısında iki kat büyük ve eşleşmeleri düşük. İlk 100.000 okumanın ortalama kalitesine bakalım; ipucu burada mı?

In [4]:
def ortalama_kalite(dosya, n=100_000):
    toplam = adet = 0
    with gzip.open(dosya, 'rt') as f:
        for i in range(n):
            f.readline(); f.readline(); f.readline()
            q = f.readline().rstrip()
            if not q: break
            toplam += sum(ord(c) - 33 for c in q)
            adet += len(q)
    return toplam / adet

for d in ['SRR384977_1M.fastq.gz', 'SRR384980_1M.fastq.gz']:
    print(d, '-> ortalama kalite:', round(ortalama_kalite(d), 1))

SRR384977_1M.fastq.gz -> ortalama kalite: 30.0
SRR384980_1M.fastq.gz -> ortalama kalite: 33.0


In [5]:
def n_say(dosya, n=100_000):
    toplam = 0
    with gzip.open(dosya, 'rt') as f:
        for i in range(n):
            f.readline(); s = f.readline(); f.readline(); f.readline()
            toplam += s.count('N')
    return toplam

for d in ['SRR384977_1M.fastq.gz', 'SRR384980_1M.fastq.gz']:
    print(d, '-> N sayısı:', n_say(d))

SRR384977_1M.fastq.gz -> N sayısı: 100841
SRR384980_1M.fastq.gz -> N sayısı: 9379


## Kendin dene

Aynı karşılaştırmayı SRR384982 (öbür şişman dosya) için yapın. Bir de her dosyada N harfi (cihazın "okuyamadım" demesi) sayın: dizi satırlarında `satir.count('N')` toplayın. Bulduklarınızı not edin; bir sonraki rehberde FastQC aynı soruya profesyonel araçla bakacak.